# 01 - Filtrar Chihuahua desde el dataset nacional consolidado
Insumo: data/processed/suicidio_2019_2024_consolidado.csv del repo
mexico-suicide-data-curation (49,918 registros, ya validado).

Este notebook asume que copiaste ese archivo a data/raw/ de este repo,
o ajustas la ruta abajo para apuntar al repo original en tu maquina.

In [ ]:
import pandas as pd

# Ajustar ruta segun donde tengas el consolidado nacional
RUTA_CONSOLIDADO_NACIONAL = '../data/raw/suicidio_2019_2024_consolidado.csv'

df_nacional = pd.read_csv(RUTA_CONSOLIDADO_NACIONAL, encoding='utf-8', low_memory=False, dtype=str)
print(f'Registros nacionales: {len(df_nacional):,}')


## 1. Filtrar a Chihuahua (Ent_ocurr == '08')

In [ ]:
df_chih = df_nacional[df_nacional['Ent_ocurr'] == '08'].copy()
print(f'Registros de Chihuahua: {len(df_chih):,}')
df_chih['anio_dataset'].value_counts().sort_index()


## 2. Verificacion rapida contra cifras publicadas
Cifras de referencia encontradas en prensa/INEGI (verificar y documentar fuente
exacta en docs/methodology.md antes de usar en el articulo):
- 2022: tasa 11.2 (1er lugar nacional)
- 2023: 561 casos, tasa 15.5 (1er lugar nacional)
- 2024: tasa 16.4 (1er lugar nacional)

In [ ]:
casos_chih_por_anio = df_chih['anio_dataset'].value_counts().sort_index()
print('Casos Chihuahua por anio (pipeline propio):')
print(casos_chih_por_anio)
print()
print('Comparar 2023 contra cifra de prensa (561 casos) al validar.')


## 3. Casos por municipio (conteo crudo, SIN tasa todavia)
OJO: esta tabla por si sola es enganosa (favorece a municipios grandes).
Sirve solo para inspeccion inicial, no para el articulo final.

In [ ]:
casos_por_municipio = df_chih.groupby(['anio_dataset', 'Mun_ocurr']).size().reset_index(name='casos')
casos_por_municipio.sort_values(['anio_dataset', 'casos'], ascending=[True, False]).head(20)


## 4. Guardar subconjunto de Chihuahua
Handoff a 02_population_merge.ipynb: unir con poblacion municipal CONAPO
para calcular tasas.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_chih.to_csv('../data/processed/suicidio_chihuahua_2019_2024.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(df_chih):,} registros')


## 5. Hallazgos
_Documentar aqui: si el conteo total coincide razonablemente con las cifras
de prensa citadas arriba, y cualquier patron inicial observado por municipio
antes de calcular tasas._